In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Prophet import
from prophet import Prophet

# Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error

plt.style.use('seaborn-v0_8-whitegrid')

print('✅ Libraries imported!')
print('✅ Prophet ready!')

✅ Libraries imported!
✅ Prophet ready!


In [5]:
DATA_PATH = r"C:\demand_Forecasting\data" + "\\"

try:
    df = pd.read_csv(DATA_PATH + 'ca1_foods_daily.csv', parse_dates=['date'])
    print('✅ ca1_foods_daily.csv load ho gayi!')
except FileNotFoundError:
    print('⚠️  ca1_foods_daily.csv nahi mili — seedha bana raha hoon...')
    sales = pd.read_csv(DATA_PATH + 'sales_train_evaluation.csv')
    cal   = pd.read_csv(DATA_PATH + 'calendar.csv')
    id_cols  = ['id','item_id','dept_id','cat_id','store_id','state_id']
    day_cols = [c for c in sales.columns if c.startswith('d_')]
    long = sales[sales['store_id']=='CA_1'].melt(
        id_vars=id_cols, value_vars=day_cols, var_name='d', value_name='sales')
    long = long.merge(cal[['d','date']], on='d', how='left')
    long['date'] = pd.to_datetime(long['date'])
    df = long[long['cat_id']=='FOODS'].groupby('date')['sales'].sum().reset_index()
    df.to_csv(DATA_PATH + 'ca1_foods_daily.csv', index=False)
    print('✅ Data ready!')

print(f'Shape      : {df.shape}')
print(f'Date range : {df["date"].min()} → {df["date"].max()}')
df.head()

✅ ca1_foods_daily.csv load ho gayi!
Shape      : (1941, 2)
Date range : 2011-01-29 00:00:00 → 2016-05-22 00:00:00


,date,sales
0,2011-01-29,3239
1,2011-01-30,3137
2,2011-01-31,2008
3,2011-02-01,2258
4,2011-02-02,2032


In [6]:
# Prophet ka required format
prophet_df = df.rename(columns={'date': 'ds', 'sales': 'y'})

print('Prophet format:')
print(prophet_df.head())
print(f'\nColumns: {prophet_df.columns.tolist()}')  # ds aur y hona chahiye

Prophet format:
          ds     y
0 2011-01-29  3239
1 2011-01-30  3137
2 2011-01-31  2008
3 2011-02-01  2258
4 2011-02-02  2032

Columns: ['ds', 'y']
